# Silver Layer — Order Details
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.orderdetails`, applies business validations
and computes line total, writing the result to `salesflow_dev.silver.orderdetails`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Validate `Quantity > 0` |
| 2 | Validate `UnitPrice >= 0` |
| 3 | Validate `Discount` between `0` and `1` |
| 4 | Calculate `line_total` |
| 5 | Add `data_quality_status` flag |
| 6 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp, when, col, round

# Read orderdetails table from Bronze layer
df = spark.table("salesflow_dev.bronze.orderdetails")

print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Cast Numeric Columns
Ensure all numeric columns are correctly typed before applying validations.

In [0]:
# Cast to correct numeric types — inferSchema may read these as strings from CSV
df = df \
    .withColumn("Quantity",  col("Quantity").cast("integer")) \
    .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
    .withColumn("Discount",  col("Discount").cast("double"))

## 3. Business Validations
Validate core numeric fields against business rules before computing derived columns.

| Column | Rule |
|---|---|
| `Quantity` | Must be greater than `0` |
| `UnitPrice` | Must be greater than or equal to `0` |
| `Discount` | Must be between `0` and `1` (inclusive) |

In [0]:
# Add individual validation flags — useful for debugging which rule failed
df = df \
    .withColumn("quantity_valid",
        col("Quantity").isNotNull() & (col("Quantity") > 0)
    ) \
    .withColumn("unitprice_valid",
        col("UnitPrice").isNotNull() & (col("UnitPrice") >= 0)
    ) \
    .withColumn("discount_valid",
        col("Discount").isNotNull() & (col("Discount") >= 0) & (col("Discount") <= 1)
    )

# Preview validation results
print("Quantity validation:")
display(df.groupBy("quantity_valid").count())

print("\nUnit price validation:")
display(df.groupBy("unitprice_valid").count())

print("\nDiscount validation:")
display(df.groupBy("discount_valid").count())

## 4. Calculate `line_total`
Computes the total value per order line after discount.

**Formula:** `Quantity × UnitPrice × (1 − Discount)`

`line_total` is only calculated for records that pass all validations.  
Invalid records receive `null` to avoid propagating bad data into analytics.

In [0]:
# Calculate line_total only for valid records — avoids polluting Gold with bad numbers
df = df.withColumn(
    "line_total",
    when(
        col("quantity_valid") & col("unitprice_valid") & col("discount_valid"),
        round(col("Quantity") * col("UnitPrice") * (1 - col("Discount")), 2)
    ).otherwise(None)  # null for invalid records
)

## 5. Add Quality Flag
A record is `INVALID` if any of the following business rules are violated:
- `Quantity` is null or `≤ 0`
- `UnitPrice` is null or `< 0`
- `Discount` is null or outside the `[0, 1]` range
- `OrderID` or `ProductID` is null (foreign key integrity)

In [0]:
# Consolidate all validation flags into a single quality status
df = df.withColumn(
    "data_quality_status",
    when(
        col("OrderID").isNull() |
        col("ProductID").isNull() |
        (~col("quantity_valid")) |
        (~col("unitprice_valid")) |
        (~col("discount_valid")),
        "INVALID"
    ).otherwise("VALID")
)

# Drop helper validation columns — no longer needed after quality flag is set
df = df.drop("quantity_valid", "unitprice_valid", "discount_valid")

## 6. Add Processing Timestamp

In [0]:
# Capture when this record was processed in the Silver layer
df = df.withColumn("processing_timestamp", current_timestamp())

## 7. Save as Delta Table

In [0]:
# Write to Silver layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.silver.orderdetails")

print("Table saved: salesflow_dev.silver.orderdetails")

## 8. Validation

In [0]:
silver_orderdetails = spark.table("salesflow_dev.silver.orderdetails")

# Record count
print(f"Total records: {silver_orderdetails.count()}")

# Quality flag distribution
print("\nQuality flag distribution:")
display(silver_orderdetails.groupBy("data_quality_status").count())

# line_total stats — useful to catch outliers or unexpected nulls
print("\nLine total statistics:")
display(silver_orderdetails.select("line_total").summary())

# Discount distribution — validate no out-of-range values made it through
print("\nDiscount distribution:")
display(silver_orderdetails.groupBy("Discount").count().orderBy("Discount"))

# Schema
print("\nSchema:")
silver_orderdetails.printSchema()

# Sample
print("\nFirst 5 rows:")
display(silver_orderdetails.limit(5))